<a href="https://colab.research.google.com/github/DrKroll/llm-humanidades-salamanca/blob/main/notebooks/taller_llm/04_calderon_preguntas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# 04 · Preguntas sobre el teatro de Calderón: modelo base, RAG y ajuste fino

**Objetivo:** entrenar **tu propio** asistente sobre Calderón y comparar cuatro estrategias:

| | Estrategia | Qué hace |
|---|---|---|
| A | **Modelo base** | Responde con lo que «recuerda» de su entrenamiento general. |
| B | **RAG** (*Retrieval-Augmented Generation*) | Primero **busca** versos relevantes en el corpus y se los da al modelo junto con la pregunta. |
| C | **SFT** (ajuste fino con LoRA) | Entrenas el modelo con ~2000 preguntas y respuestas generadas automáticamente a partir del TEI. |
| D | **SFT + RAG** | El modelo ajustado recibe además los fragmentos encontrados. |

**Corpus:** 166 obras de Calderón en TEI-XML (89 comedias y dramas, 77 autos sacramentales y loas), sin duplicados,
de los corpus TEI de Simon Kroll (EMOTHE/ARTELOPE, DraCor, ediciones propias...). Fuentes: `data/FUENTES.md`.
El entrenamiento usa una muestra de ~2000 preguntas de todas las obras, así que dura lo mismo que con pocas obras.

⏱️ **≈ 30 min** (martes 16:30–17:00). El entrenamiento en tu GPU T4 dura ≈ 4–7 min (como máximo 10: después se para solo).

In [ ]:
#@title ⚙️ Preparación: ejecuta esta celda primero (no hace falta leer el código) { display-mode: "form" }
# ⚙️ Preparación: ejecuta esta celda primero (Mayús + Intro). Puedes repetirla sin problema.
import os, sys, importlib.util
REPO_URL = "https://github.com/DrKroll/llm-humanidades-salamanca"
REPO_DIR = "/content/llm-humanidades-salamanca"

if "google.colab" in sys.modules:
    # En Colab: descargamos el repositorio (solo la primera vez) y entramos en su carpeta
    if not os.path.exists(REPO_DIR):
        !git clone -q {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)
    # Colab ya trae torch, transformers, peft, etc.: solo instalamos lo que falte,
    # sin tocar las versiones preinstaladas
    paquetes = {"transformers": "transformers", "tokenizers": "tokenizers", "peft": "peft",
                "accelerate": "accelerate", "yaml": "pyyaml", "networkx": "networkx", "tqdm": "tqdm"}
    faltan = [pip for mod, pip in paquetes.items() if importlib.util.find_spec(mod) is None]
    if faltan:
        !pip install -q {" ".join(faltan)}
else:
    # En tu ordenador: subimos de carpeta hasta encontrar config.yaml
    while not os.path.exists("config.yaml") and os.getcwd() != "/":
        os.chdir("..")
# El código del taller (carpeta src/dhllm) se importa directamente, sin reiniciar
if os.path.abspath("src") not in sys.path:
    sys.path.insert(0, os.path.abspath("src"))
print("Carpeta de trabajo:", os.getcwd())

## 0. ¿Tenemos GPU?

In [ ]:
from dhllm import taller
taller.comprobar_gpu();

## ⚙️ Ajustes
Esto es todo lo que hace falta tocar. `"obras"` puede ser `"todas"` o una lista de títulos, p. ej.
`["La vida es sueño", "Eco y Narciso"]`. El resto está en `config.yaml` (sección `calderon`).

In [ ]:
AJUSTES = {"epocas": 1, "ritmo_aprendizaje": 2e-4, "obras": "todas", "max_minutos": 10}
USAR_COPIA = False   # True = sin GPU, si falla o si no hay tiempo: usar el adaptador ya entrenado (backups/)
PREGUNTAS = taller.PREGUNTAS_CALDERON   # (pregunta, respuesta esperada según el corpus)

## 1. Preparar los datos
Leemos el TEI (jornadas, personajes `<sp>`/`<speaker>`, versos `<l>`, acotaciones `<stage>`), construimos un
**buscador** (BM25, la técnica clásica de los buscadores) y generamos preguntas con respuesta **verificable**:
reparto, papel de cada personaje, quién habla más, quién dice unos versos, de qué obra son...

In [ ]:
datos = taller.preparar_datos(AJUSTES)

In [ ]:
taller.ver_obra(datos, "Eco y Narciso")

## 2. Cargar el modelo base (Qwen2.5-0.5B-Instruct)

In [ ]:
modelo = taller.cargar_modelo(datos)

## 3. (A) Preguntar al modelo base
Fíjate en si las respuestas son **correctas** o solo **suenan bien**. Un modelo de lenguaje siempre
produce un texto plausible, aunque no «sepa» la respuesta: eso es una **alucinación**.

In [ ]:
A = taller.responder(modelo, PREGUNTAS)

## 4. (B) RAG: buscar primero, responder después
Para cada pregunta buscamos los fragmentos (~12 versos) más parecidos y se los damos al modelo. Así ve el buscador una pregunta:

In [ ]:
taller.mostrar_busqueda(datos, PREGUNTAS[0][0])

In [ ]:
B = taller.responder(modelo, PREGUNTAS, datos)

## 5. (C) Entrenar tu modelo (ajuste fino con LoRA)
El modelo ve cada pregunta y aprende a escribir la respuesta. Algunas preguntas llevan fragmentos de RAG
para que aprenda también a usarlos. Verás una **barra de progreso**; la pérdida debe bajar.

🛟 Si no hay GPU, si algo falla o si pones `USAR_COPIA = True`, se carga automáticamente una copia ya entrenada.

In [ ]:
modelo = taller.entrenar_calderon(modelo, datos, usar_copia=USAR_COPIA)

## 6. (C) y (D): preguntar al modelo ajustado, sin y con RAG

In [ ]:
C = taller.responder(modelo, PREGUNTAS)
D = taller.responder(modelo, PREGUNTAS, datos)

## 7. Comparación
Lee la tabla con ojos de filólogo/a: ¿qué columna acierta?, ¿cuál inventa?, ¿cuál cita?

In [ ]:
taller.tabla_comparativa(PREGUNTAS, A=A, B=B, C=C, D=D)

Haz tu propia pregunta (con `datos` = con RAG; sin `datos` = sin RAG):

In [ ]:
taller.preguntar(modelo, "¿Quién es Narciso en «Eco y Narciso»?", datos);

> ⭐ **Opcional** — puedes saltarte esta sección si vas justo de tiempo.

### ¿Y con preguntas que el modelo **no** vio al entrenar?
Estos pares se generaron igual que los de entrenamiento, pero los reservamos. Comparamos el modelo base,
el ajustado y el ajustado con RAG.

In [ ]:
taller.evaluar_preguntas_nuevas(modelo, datos, n=6)

> ⭐ **Opcional** (5–10 min) — puedes saltarte esta sección si vas justo de tiempo.

## 8. Grafo de personajes: la estructura de la obra

El buscador encuentra **versos**, pero le cuesta responder a preguntas sobre la **estructura**: ¿quién es el
Capitán?, ¿con quién habla más Segismundo? Para eso las Humanidades Digitales usan el **análisis de redes**
(Franco Moretti, *Network Theory, Plot Analysis*, 2011; el proyecto [DraCor](https://dracor.org)).

Lo construimos directamente del TEI:
- **nodo** = un personaje, con su **papel** según el reparto (`<castList>`) y su número de **versos**;
- **arista** = «dialoga con»: dos parlamentos seguidos en la misma jornada. El **peso** es el número de réplicas.

Es una aproximación: hablar justo después de alguien no siempre es hablarle *a* él.

In [ ]:
G = taller.grafo(datos, "La vida es sueño")

### Dibujar la red
Tamaño del círculo = versos; grosor de la línea = réplicas.

In [ ]:
taller.dibujar_grafo(G)

### ¿Quién es el personaje más central?
No hay una sola respuesta: depende de **qué** medimos. *Interlocutores* = con cuántos personajes distintos habla;
*réplicas* = cuánto diálogo intercambia; *intermediación* = cuántas veces está «en medio» entre otros dos.

In [ ]:
taller.personajes_centrales(G);

### Preguntar con el grafo **y** con los versos
Ahora el modelo recibe dos fuentes: los **datos del grafo** (estructura: papel, versos, con quién habla)
y los **versos del buscador** (citas con jornada y número de verso). Comparamos con la columna D (solo versos).

In [ ]:
taller.comparar_con_grafo(modelo, datos)   # preguntas: taller.PREGUNTAS_GRAFO

**Para pensar:** el **grafo aporta la estructura** (quién es quién, cuánto habla, con quién); la **búsqueda
aporta las citas** (qué dice exactamente y dónde). Ninguna de las dos fuentes basta sola.
- El grafo solo sabe lo que está **marcado** en el TEI: si el reparto no da el papel, el modelo tampoco lo sabrá.
- Prueba `taller.dibujar_grafo(taller.grafo(datos, "El alcalde de Zalamea"))`: ¿quién hace de «puente» entre los soldados y los labradores?

## 💬 Para discutir: *el ajuste fino cambia el comportamiento y el estilo; el RAG aporta los hechos*

- El **modelo base** inventa con seguridad: nombres, tramas, fechas.
- El **RAG** encuentra los versos reales; el modelo los resume y puede **citar** el fragmento. Si el buscador
  no encuentra el pasaje correcto (pregunta sobre la *trama*, p. ej. Segismundo en la torre), el modelo
  tampoco acierta: la calidad depende de la búsqueda.
- El **SFT** enseña el **formato** («Los dice X, en la jornada N») y algunos datos que se repitieron mucho, pero
  no puede memorizar 40 000 versos: con citas nuevas suele adivinar un personaje plausible.
- **SFT + RAG** suele ser lo mejor: el formato aprendido + los hechos del corpus.
- ⭐ El **grafo de personajes** completa al RAG con la **estructura** de la obra (papeles, quién habla con quién): los datos del TEI, no la memoria del modelo.
- ⚠️ El modelo solo «conoce» **esta edición**: en la de EMOTHE, *La vida es sueño* empieza con «Hipogrifo
  *triste*», no «violento». ¿Qué implica esto para la crítica textual y para usar LLM como fuente?

> 🔍 **¿Quieres mirar el código?** [`src/dhllm/taller.py`](../../src/dhllm/taller.py) (sección 3) llama a
> [`src/dhllm/calderon.py`](../../src/dhllm/calderon.py) (TEI, buscador, preguntas automáticas, grafo) y a
> [`src/dhllm/sft.py`](../../src/dhllm/sft.py) (entrenamiento con pares pregunta-respuesta).

## 🖐️ Tu turno
1. Haz tus propias preguntas con `taller.preguntar(...)` (sobre el reparto, sobre citas, sobre la trama).
2. Entrena solo con dos obras: `"obras": ["La vida es sueño", "Eco y Narciso"]`. ¿Mejora en esas obras?
3. Busca con `datos.indice.search("honor", k=5)` o con otra palabra clave: ¿en qué obras aparece?
4. **Para pensar:** ¿podríamos publicar este modelo ajustado? ¿Quién tiene derechos sobre una edición crítica?